# NeRF：Neural Radiance Fields 三维场景重建

这个 Notebook 从零实现 `NeRF（Neural Radiance Fields）`，用多视角图像隐式表达三维场景。

内容包括：
- 体渲染方程与 NeRF 的联系
- 位置编码（Positional Encoding）的作用
- 分层采样（Coarse-to-Fine）策略
- 微型 NeRF 从零实现（2D/3D 场景）
- 体素射线采样可视化
- 新视角合成结果展示
- 与 3D Gaussian Splatting 的对比

## 1. 环境准备

```bash
pip install torch torchvision matplotlib numpy tqdm
```

In [ ]:
import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 渲染图像分辨率
    H: int = 64
    W: int = 64
    # 相机焦距（影响视野角）
    focal: float = 50.0
    # 每条射线采样点数
    n_samples: int = 64
    # 场景近远裁切面
    near: float = 2.0
    far: float = 6.0
    # 位置编码频率阶数
    L_pos: int = 10
    L_dir: int = 4
    # MLP 隐层维度
    hidden_dim: int = 256
    lr: float = 5e-4
    n_iters: int = 2000
    # 每次迭代随机采样射线数
    n_rays: int = 1024

cfg = Config()
cfg

## 2. NeRF 核心原理

### 2.1 场景表达方式对比

| 方法 | 表达形式 | 内存 | 分辨率 |
|------|---------|------|--------|
| 体素网格 | 显式 3D 数组 | 大（分辨率³） | 受限于网格精度 |
| 点云 | 显式离散点 | 中 | 稀疏 |
| 网格 | 显式三角面 | 小 | 取决于面数 |
| **NeRF** | **隐式 MLP** | **极小（网络参数）** | **连续，任意分辨率** |

### 2.2 NeRF 的核心函数

NeRF 用一个 MLP 表示整个场景：

$$F_\theta(\mathbf{x}, \mathbf{d}) \rightarrow (\mathbf{c}, \sigma)$$

- $\mathbf{x} = (x, y, z)$：三维空间坐标
- $\mathbf{d} = (\theta, \phi)$：观察方向（方位角 + 仰角）
- $\mathbf{c} = (r, g, b)$：该点在方向 $\mathbf{d}$ 下的颜色（与观察方向相关，实现镜面反射等效果）
- $\sigma$：体积密度（与观察方向无关，由几何决定）

### 2.3 体渲染方程

从相机出发射出一条射线 $\mathbf{r}(t) = \mathbf{o} + t\mathbf{d}$，像素颜色为：

$$C(\mathbf{r}) = \int_{t_n}^{t_f} T(t) \cdot \sigma(\mathbf{r}(t)) \cdot \mathbf{c}(\mathbf{r}(t), \mathbf{d}) \, dt$$

$$T(t) = \exp\!\left(-\int_{t_n}^{t} \sigma(\mathbf{r}(s)) \, ds\right)$$

$T(t)$ 是累积透射率：射线到达 $t$ 点前未被遮挡的概率。

### 2.4 离散化近似

$$\hat{C} = \sum_{i=1}^{N} T_i (1 - e^{-\sigma_i \delta_i}) \mathbf{c}_i, \quad T_i = \exp\!\left(-\sum_{j<i} \sigma_j \delta_j\right)$$

## 3. 位置编码（Positional Encoding）

In [ ]:
def positional_encoding(x, L):
    # 高频正弦/余弦编码，让 MLP 能学习高频细节
    # 不编码的话 MLP 倾向于过度平滑，丢失纹理
    freqs = 2.0 ** torch.arange(L, dtype=x.dtype, device=x.device)  # [L]
    x_freq = x[..., None] * freqs  # [..., D, L]
    x_freq = x_freq.reshape(*x.shape[:-1], -1)  # [..., D*L]
    return torch.cat([x, torch.sin(x_freq), torch.cos(x_freq)], dim=-1)


# 可视化：用高频编码 vs 不编码，拟合同一个 1D 高频信号的差异
t = torch.linspace(0, 1, 200)
signal = torch.sin(2 * math.pi * 8 * t)  # 8Hz 目标信号

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, use_pe, title in zip(axes, [False, True], ['无位置编码（MLP 直接吃坐标）', '有位置编码（L=6）']):
    L = 6 if use_pe else 0
    in_dim = 1 + 2 * 1 * L if use_pe else 1
    net = nn.Sequential(nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
    opt = optim.Adam(net.parameters(), lr=1e-3)

    for _ in range(500):
        inp = t.unsqueeze(-1)
        if use_pe:
            inp = positional_encoding(inp, L)
        pred = net(inp).squeeze(-1)
        loss = ((pred - signal) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()

    with torch.no_grad():
        inp = t.unsqueeze(-1)
        if use_pe:
            inp = positional_encoding(inp, L)
        out = net(inp).squeeze(-1).numpy()

    ax.plot(t.numpy(), signal.numpy(), label='目标信号', linewidth=2)
    ax.plot(t.numpy(), out, '--', label='MLP 拟合', linewidth=2)
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

## 4. NeRF MLP 实现

In [ ]:
class NeRF(nn.Module):
    def __init__(self, L_pos=10, L_dir=4, hidden_dim=256):
        super().__init__()
        # 位置编码后的输入维度：3 + 2*3*L_pos
        pos_dim = 3 + 2 * 3 * L_pos
        # 方向编码后的输入维度：3 + 2*3*L_dir
        dir_dim = 3 + 2 * 3 * L_dir

        # 主干：处理位置信息，输出体积密度和中间特征
        # skip connection 在第 4 层重新输入位置编码，防止梯度消失
        self.block1 = nn.Sequential(
            nn.Linear(pos_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.block2 = nn.Sequential(
            # 第 5 层 skip：将位置编码重新拼入
            nn.Linear(hidden_dim + pos_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        # 体积密度：只由位置决定，不受观察方向影响
        self.sigma_head = nn.Linear(hidden_dim, 1)
        # 特征向量传递给颜色分支
        self.feature_head = nn.Linear(hidden_dim, hidden_dim)
        # 颜色：由位置特征 + 观察方向共同决定（非朗伯体表面）
        self.color_block = nn.Sequential(
            nn.Linear(hidden_dim + dir_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, 3), nn.Sigmoid(),
        )
        self.L_pos = L_pos
        self.L_dir = L_dir

    def forward(self, pts, dirs):
        pts_enc  = positional_encoding(pts,  self.L_pos)
        dirs_enc = positional_encoding(dirs, self.L_dir)

        h = self.block1(pts_enc)
        h = self.block2(torch.cat([h, pts_enc], dim=-1))

        sigma = torch.relu(self.sigma_head(h))  # 密度非负
        feat  = self.feature_head(h)
        color = self.color_block(torch.cat([feat, dirs_enc], dim=-1))

        return color, sigma


model = NeRF(cfg.L_pos, cfg.L_dir, cfg.hidden_dim).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'NeRF 参数量：{total_params:,}')

## 5. 射线生成与体渲染

In [ ]:
def get_rays(H, W, focal, c2w):
    # 生成图像平面上每个像素对应的射线方向（相机坐标系 → 世界坐标系）
    i, j = torch.meshgrid(
        torch.arange(W, dtype=torch.float32),
        torch.arange(H, dtype=torch.float32),
        indexing='xy'
    )
    dirs = torch.stack([
        (i - W * 0.5) / focal,
        -(j - H * 0.5) / focal,
        -torch.ones_like(i)
    ], dim=-1)  # H, W, 3

    # 旋转到世界坐标系
    rays_d = (dirs[..., None, :] * c2w[:3, :3]).sum(dim=-1)  # H, W, 3
    rays_o = c2w[:3, 3].expand_as(rays_d)
    return rays_o, rays_d


def volume_render(rgb, sigma, z_vals):
    # 相邻采样点间距
    deltas = z_vals[..., 1:] - z_vals[..., :-1]
    # 最后一段延伸到无穷远，确保累积透射率归零
    deltas = torch.cat([deltas, torch.full_like(deltas[..., :1], 1e10)], dim=-1)

    alpha  = 1.0 - torch.exp(-sigma[..., 0] * deltas)  # 每段的不透明度
    T      = torch.cumprod(torch.cat([torch.ones_like(alpha[..., :1]), 1.0 - alpha + 1e-10], dim=-1), dim=-1)[..., :-1]
    weights = T * alpha  # 每个采样点对最终颜色的贡献权重

    rgb_map   = (weights[..., None] * rgb).sum(dim=-2)
    depth_map = (weights * z_vals).sum(dim=-1)
    return rgb_map, depth_map, weights


print('射线生成与体渲染函数定义完成')

## 6. 合成数据集生成（微型球体场景）

In [ ]:
def generate_spherical_poses(n_views=20, radius=4.0):
    # 在球面上均匀分布相机位置，俯仰角 30°
    poses = []
    for i in range(n_views):
        theta = 2 * math.pi * i / n_views
        phi   = math.pi / 6  # 30 度俯仰

        cam_pos = torch.tensor([
            radius * math.cos(phi) * math.cos(theta),
            radius * math.cos(phi) * math.sin(theta),
            radius * math.sin(phi),
        ], dtype=torch.float32)

        # 计算 look-at 变换
        z = -cam_pos / cam_pos.norm()  # 朝向原点
        up = torch.tensor([0, 0, 1], dtype=torch.float32)
        x = torch.cross(up, z)
        x = x / (x.norm() + 1e-8)
        y = torch.cross(z, x)

        c2w = torch.stack([x, y, z, cam_pos], dim=-1)  # 3x4
        c2w = torch.cat([c2w, torch.tensor([[0, 0, 0, 1]], dtype=torch.float32)], dim=0)  # 4x4
        poses.append(c2w)
    return torch.stack(poses)


def render_gt_sphere(H, W, focal, c2w, sphere_center=(0, 0, 0), sphere_r=1.0):
    # 用解析射线-球体求交生成 GT 图像（红色球体）
    rays_o, rays_d = get_rays(H, W, focal, c2w)
    rays_o = rays_o.reshape(-1, 3)
    rays_d = rays_d.reshape(-1, 3)
    rays_d = rays_d / rays_d.norm(dim=-1, keepdim=True)

    center = torch.tensor(sphere_center, dtype=torch.float32)
    oc = rays_o - center
    a = (rays_d * rays_d).sum(-1)
    b = 2.0 * (oc * rays_d).sum(-1)
    c = (oc * oc).sum(-1) - sphere_r ** 2
    disc = b * b - 4 * a * c

    hit = disc >= 0
    img = torch.ones(H * W, 3) * 0.9  # 背景浅灰色

    if hit.any():
        t_hit = (-b[hit] - torch.sqrt(disc[hit].clamp(0))) / (2 * a[hit])
        p_hit = rays_o[hit] + t_hit[:, None] * rays_d[hit]
        normal = (p_hit - center) / sphere_r
        # 简单漫反射着色：法线 * 颜色
        shading = (normal * torch.tensor([0.6, 0.8, 1.0])).clamp(0, 1)
        img[hit] = shading

    return img.reshape(H, W, 3)


poses = generate_spherical_poses(n_views=25)
images_gt = [render_gt_sphere(cfg.H, cfg.W, cfg.focal, pose) for pose in poses]

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, img in zip(axes, images_gt[:5]):
    ax.imshow(img.numpy())
    ax.axis('off')
plt.suptitle('GT 多视角图像（合成球体场景）')
plt.tight_layout()
plt.show()

## 7. 训练主循环

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9999)
losses = []

for iteration in range(cfg.n_iters):
    # 随机选一个视角
    view_idx = np.random.randint(len(poses))
    c2w = poses[view_idx].to(device)
    gt  = images_gt[view_idx].to(device)

    # 生成该视角所有射线
    rays_o, rays_d = get_rays(cfg.H, cfg.W, cfg.focal, c2w)
    rays_o = rays_o.reshape(-1, 3)
    rays_d = rays_d.reshape(-1, 3)

    # 随机采样子集射线，减少显存占用
    sel = torch.randperm(rays_o.shape[0])[:cfg.n_rays]
    rays_o_sel = rays_o[sel]
    rays_d_sel = rays_d[sel]
    gt_sel     = gt.reshape(-1, 3)[sel]

    # 在每条射线上均匀采样 N 个点（加小随机扰动防止 alias）
    t_vals = torch.linspace(cfg.near, cfg.far, cfg.n_samples, device=device)
    noise  = torch.rand_like(t_vals) * (cfg.far - cfg.near) / cfg.n_samples
    t_vals = t_vals + noise

    pts  = rays_o_sel[:, None, :] + t_vals[None, :, None] * rays_d_sel[:, None, :]
    dirs = rays_d_sel[:, None, :].expand_as(pts)
    dirs = dirs / (dirs.norm(dim=-1, keepdim=True) + 1e-8)

    pts_flat  = pts.reshape(-1, 3)
    dirs_flat = dirs.reshape(-1, 3)

    rgb_flat, sigma_flat = model(pts_flat, dirs_flat)
    rgb   = rgb_flat.reshape(cfg.n_rays, cfg.n_samples, 3)
    sigma = sigma_flat.reshape(cfg.n_rays, cfg.n_samples, 1)

    rgb_map, _, _ = volume_render(rgb, sigma, t_vals.expand(cfg.n_rays, -1))
    loss = ((rgb_map - gt_sel) ** 2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    losses.append(loss.item())

    if (iteration + 1) % 200 == 0:
        print(f'Iter {iteration+1:5d}/{cfg.n_iters}  loss={loss.item():.6f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses)
ax.set_title('NeRF 训练损失')
ax.set_xlabel('Iteration')
ax.set_ylabel('MSE Loss')
plt.tight_layout()
plt.show()

## 8. 新视角合成展示

In [ ]:
@torch.no_grad()
def render_full_image(model, H, W, focal, c2w, cfg, device):
    model.eval()
    rays_o, rays_d = get_rays(H, W, focal, c2w.to(device))
    rays_o = rays_o.reshape(-1, 3)
    rays_d = rays_d.reshape(-1, 3)

    chunk = 1024
    rgb_chunks = []
    depth_chunks = []

    for i in range(0, rays_o.shape[0], chunk):
        ro = rays_o[i:i+chunk]
        rd = rays_d[i:i+chunk]
        t_vals = torch.linspace(cfg.near, cfg.far, cfg.n_samples, device=device)
        pts  = ro[:, None, :] + t_vals[None, :, None] * rd[:, None, :]
        dirs = rd[:, None, :].expand_as(pts)
        dirs = dirs / (dirs.norm(dim=-1, keepdim=True) + 1e-8)

        rgb_f, sigma_f = model(pts.reshape(-1, 3), dirs.reshape(-1, 3))
        rgb   = rgb_f.reshape(ro.shape[0], cfg.n_samples, 3)
        sigma = sigma_f.reshape(ro.shape[0], cfg.n_samples, 1)

        rgb_map, depth_map, _ = volume_render(rgb, sigma, t_vals.expand(ro.shape[0], -1))
        rgb_chunks.append(rgb_map.cpu())
        depth_chunks.append(depth_map.cpu())

    rgb_img   = torch.cat(rgb_chunks).reshape(H, W, 3).clamp(0, 1)
    depth_img = torch.cat(depth_chunks).reshape(H, W)
    return rgb_img.numpy(), depth_img.numpy()


# 渲染训练中未见过的新视角
novel_poses = generate_spherical_poses(n_views=4)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, pose in enumerate(novel_poses):
    gt_img   = render_gt_sphere(cfg.H, cfg.W, cfg.focal, pose)
    pred_img, depth_img = render_full_image(model, cfg.H, cfg.W, cfg.focal, pose, cfg, device)

    axes[0, i].imshow(gt_img.numpy())
    axes[0, i].set_title(f'GT 视角 {i+1}')
    axes[0, i].axis('off')

    axes[1, i].imshow(pred_img)
    axes[1, i].set_title(f'NeRF 渲染')
    axes[1, i].axis('off')

plt.suptitle('NeRF 新视角合成（上：GT，下：NeRF 渲染）')
plt.tight_layout()
plt.show()

## 9. NeRF vs 3D Gaussian Splatting

| 维度 | NeRF | 3D Gaussian Splatting |
|------|------|-----------------------|
| 场景表达 | 隐式 MLP | 显式 3D 高斯椭球 |
| 训练时间 | 数小时（原始 NeRF） | 数分钟（3DGS） |
| 渲染速度 | 慢（每帧需跑 MLP 数百次） | 实时（光栅化投影） |
| 内存 | 小（网络权重） | 中（高斯点云） |
| 编辑性 | 难（隐式，不透明） | 好（高斯是显式结构，可直接操作） |
| 无边界场景 | 需要特殊处理（mip-NeRF 360） | 处理更自然 |

**NeRF 的后续改进**：Instant-NGP（哈希编码，训练提速 100x）、mip-NeRF、Zip-NeRF 等大幅改善了原始 NeRF 的缺点。